In [3]:
import Gmsh: gmsh
import Pkg; Pkg.add("Distributions"); Pkg.add("FFTW"); Pkg.add("SpecialFunctions");

using Gridap, GridapGmsh
using Random, LinearAlgebra, FFTW
using Distributions
using Base.Threads
using Plots 
using SpecialFunctions #besselk, gamma functions


JULIA_NUM_THREADS=12
BLAS.set_num_threads(1)

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`


In [4]:
function create_square_model(h)
    gmsh.initialize()
    gmsh.model.add("unit_square")
    gmsh.model.geo.addPoint(0,0,0,1,1) # last argument is optional identifier, unique per dimension
    gmsh.model.geo.addPoint(1,0,0,1,2)
    gmsh.model.geo.addPoint(1,1,0,1,3)
    gmsh.model.geo.addPoint(0,1,0,1,4)
    gmsh.model.geo.addLine(1,2,1)
    gmsh.model.geo.addLine(2,3,2) # line 2 goes from point 2 to point 3
    gmsh.model.geo.addLine(3,4,3)
    gmsh.model.geo.addLine(4,1,4)
        
    gmsh.model.geo.addCurveLoop([1,2,3,4],1)
        
    gmsh.model.geo.addPlaneSurface([1],1)

    N = Int(round(1/h))  # number of elements per edge

    gmsh.model.geo.mesh.setTransfiniteCurve(1, N+1)
    gmsh.model.geo.mesh.setTransfiniteCurve(2, N+1)
    gmsh.model.geo.mesh.setTransfiniteCurve(3, N+1)
    gmsh.model.geo.mesh.setTransfiniteCurve(4, N+1)

    #triangulation
    gmsh.model.geo.mesh.setTransfiniteSurface(1)

    gmsh.model.geo.synchronize()

    # Define physical groups without the string argument
    edges_tag = gmsh.model.addPhysicalGroup(1, [1, 2, 3, 4])   # edges
    corners_tag = gmsh.model.addPhysicalGroup(0, [1, 2, 3, 4]) # corners
    domain_tag = gmsh.model.addPhysicalGroup(2, [1])           # surface
    
    # Set names for the physical groups
    gmsh.model.setPhysicalName(1, edges_tag, "boundary")
    gmsh.model.setPhysicalName(0, corners_tag, "boundary")
    gmsh.model.setPhysicalName(2, domain_tag, "domain")
    gmsh.model.mesh.generate(2)

    model = GmshDiscreteModel(gmsh);
    gmsh.finalize();
    return model
end



create_square_model (generic function with 1 method)

In [5]:
rho_exp(r, nu, p) = nu^2 * exp(-p*r)

function rho_matern(d, σ, nu, p)
    if d == 0
        return σ^2
    end
    return σ^2 * (2.0^(1-nu) / gamma(nu)) * (sqrt(2*nu) * d / p)^nu *
           besselk(nu, (sqrt(2*nu) * d / p))
end

function algorithm1(m0, σ, nu, p)
    m  = m0
    h0 = 1 / m0
    
    while true
        ℓ = m * h0
        n = 2m
        s = n^2
        r = zeros(Float64, s)
        
        for k1 in 1:n, k2 in 1:n
            x1 = h0 * (k1-1)
            x2 = h0 * (k2-1)
            
            φ1 = x1 < ℓ ? x1 : 2ℓ - x1
            φ2 = x2 < ℓ ? x2 : 2ℓ - x2

            idx = k1 + (k2-1)*n
            r[idx] = rho_matern(sqrt(φ1^2 + φ2^2), σ, nu, p)
        end

        λ = real.(fft(reshape(r, n, n)))

        if minimum(λ) >= 0
            return m, λ
        end

        m += 1
    end
end

function algorithm2(m0, m, λ, rng)
    n = 2m
    s = n^2
    y = randn(rng, n, n)

    y .*= sqrt.(λ)
    w = fft(y) / sqrt(s)
    w = real.(w) .+ imag.(w)

    z = w[1:(m0+1), 1:(m0+1)]
    return z
end

algorithm2 (generic function with 1 method)

In [6]:
function ce_nodal_value(Z, h, x)
    m0 = size(Z,1) - 1

    i = clamp(Int(round(x[1] / h)) + 1, 1, m0 + 1)
    j = clamp(Int(round(x[2] / h)) + 1, 1, m0 + 1)

    return Z[i,j]
end

function noise_load_from_grid(Z, Δt, ce_h, Unoise, Bnoise)
    # sample on CE grid, then interpolate onto FE grid through Unoise
    η(x) = sqrt(Δt) * ce_nodal_value(Z, ce_h, x)
    ηh = interpolate_everywhere(η, Unoise)
    zvec = get_free_dof_values(ηh)
    return Bnoise * zvec
end

function build_reference_noise_loads(T, Nref, m0, m, λ, ce_h, Unoise, Bnoise, rng)
    Δtref = T / Nref
    noise_ref = Vector{Vector{Float64}}(undef, Nref)

    for n in 1:Nref
        Z = algorithm2(m0, m, λ, rng)
        noise_ref[n] = noise_load_from_grid(Z, Δtref, ce_h, Unoise, Bnoise)
    end

    return noise_ref
end

build_reference_noise_loads (generic function with 1 method)

In [7]:
h = 2^-5
model = create_square_model(h)

order = 1
reffe = ReferenceFE(lagrangian, Float64, order)

V = TestFESpace(model, reffe, conformity=:H1, dirichlet_tags="boundary")
U = TrialFESpace(V, 0.0)

# extra space ONLY for representing the sampled CE field
Vnoise = TestFESpace(model, reffe, conformity=:H1)
Unoise = TrialFESpace(Vnoise)

Ω = Triangulation(model)
dΩ = Measure(Ω, 2 * order)

n = 8
Δt, T = 2.0^(-n), 1
num_steps = Int(round(T / Δt))

u₀ = x -> 0.0
v₀ = x -> 0.0

m(u, v) = ∫(u*v)dΩ
k(u, v) = ∫(∇(u) ⋅ ∇(v))dΩ

Mmat = assemble_matrix(m, U, V)
Kmat = assemble_matrix(k, U, V)

# maps CE field coefficients to FE load vector
Bnoise = assemble_matrix(m, Unoise, V)

LHSpos = (2/Δt)*Mmat + (Δt/2)*Kmat
RHSpos = (2/Δt)*Mmat - (Δt/2)*Kmat

uh0 = interpolate_everywhere(u₀, U)
vh0 = interpolate_everywhere(v₀, U)

# CE parameters
σ = 1.0
nu = 0.4
p = 0.15

ce_h = h
m0 = Int(round(1/ce_h))
m_ce, λ_ce = algorithm1(m0, σ, nu, p)

Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 30%] Meshing curve 2 (Line)
Info    : [ 60%] Meshing curve 3 (Line)
Info    : [ 80%] Meshing curve 4 (Line)
Info    : Done meshing 1D (Wall 0.00073521s, CPU 0.00025s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Transfinite)
Info    : Done meshing 2D (Wall 0.00350559s, CPU 0.001021s)
Info    : 1089 nodes 2180 elements


(32, [143.44013499597 103.10110301732414 … 50.63133871317729 103.10110301732414; 103.10110301732414 78.50457351249582 … 42.76428021532057 78.50457351249582; … ; 50.6313387131773 42.764280215320554 … 28.213897894446525 42.764280215320554; 103.10110301732414 78.50457351249582 … 42.76428021532057 78.50457351249582])

In [8]:
if !isdir("tmp") mkdir("tmp") end

rng_path = Xoshiro(2026)

# just to have an eta field also at t=0 in the first file
Z0 = algorithm2(m0, m_ce, λ_ce, rng_path)
η0(x) = sqrt(Δt) * ce_nodal_value(Z0, ce_h, x)
ηh0 = interpolate_everywhere(η0, Unoise)

createpvd("wave") do pvd
    pvd[0.0] = createvtk(
        Ω,
        "tmp/stoc_wavev3_0.vtu",
        cellfields = [
            "uh"  => uh0,
            "vh"  => vh0,
            "eta" => ηh0
        ]
    )

    uvec = get_free_dof_values(uh0)
    vvec = get_free_dof_values(vh0)
    
    for n in 1:num_steps
        t_n = n * Δt

        Z = algorithm2(m0, m_ce, λ_ce, rng_path)

        η(x) = sqrt(Δt) * ce_nodal_value(Z, ce_h, x)
        ηh = interpolate_everywhere(η, Unoise)

        zvec = get_free_dof_values(ηh)
        noise = Bnoise * zvec

        rhs_u = RHSpos * uvec + (2*Mmat) * vvec + 2 * noise
        uvec_new = LHSpos \ rhs_u

        rhs_v = Mmat * vvec - (Δt/2) * Kmat * (uvec_new + uvec) + noise
        vvec_new = Mmat \ rhs_v

        uh = FEFunction(U, uvec_new)
        vh = FEFunction(U, vvec_new)

        vtufilename = "tmp/stoc_wavev3_$(n).vtu"
        pvd[t_n] = createvtk(
            Ω,
            vtufilename,
            cellfields = [
                "uh"  => uh,
                "vh"  => vh,
                "eta" => ηh
            ]
        )

        uvec = uvec_new
        vvec = vvec_new
    end
end

258-element Vector{String}:
 "wave.pvd"
 "tmp/stoc_wavev3_0.vtu"
 "tmp/stoc_wavev3_1.vtu"
 "tmp/stoc_wavev3_2.vtu"
 "tmp/stoc_wavev3_3.vtu"
 "tmp/stoc_wavev3_4.vtu"
 "tmp/stoc_wavev3_5.vtu"
 "tmp/stoc_wavev3_6.vtu"
 "tmp/stoc_wavev3_7.vtu"
 "tmp/stoc_wavev3_8.vtu"
 "tmp/stoc_wavev3_9.vtu"
 "tmp/stoc_wavev3_10.vtu"
 "tmp/stoc_wavev3_11.vtu"
 ⋮
 "tmp/stoc_wavev3_245.vtu"
 "tmp/stoc_wavev3_246.vtu"
 "tmp/stoc_wavev3_247.vtu"
 "tmp/stoc_wavev3_248.vtu"
 "tmp/stoc_wavev3_249.vtu"
 "tmp/stoc_wavev3_250.vtu"
 "tmp/stoc_wavev3_251.vtu"
 "tmp/stoc_wavev3_252.vtu"
 "tmp/stoc_wavev3_253.vtu"
 "tmp/stoc_wavev3_254.vtu"
 "tmp/stoc_wavev3_255.vtu"
 "tmp/stoc_wavev3_256.vtu"

## Convergence Testing ##

In [9]:
function EMpath_stoch_time(uh0, vh0, T, Nref, kappa, facLHS, facM, Mmat, Kmat, noise_ref)
    Dtref = T / Nref
    Dt = kappa * Dtref
    NN = div(Nref, kappa)

    a = 2 / Dt
    b = Dt / 2

    uvec = copy(get_free_dof_values(uh0))
    vvec = copy(get_free_dof_values(vh0))

    uvec_new = similar(uvec)
    vvec_new = similar(uvec)

    Mu = similar(uvec)
    Ku = similar(uvec)
    Mv = similar(uvec)
    usum = similar(uvec)

    rhs_u = similar(uvec)
    rhs_v = similar(uvec)

    noise = zeros(eltype(uvec), length(uvec))

    for n in 1:NN
        fill!(noise, 0.0)
        @inbounds for r in 1:kappa
            noise .+= noise_ref[(n-1)*kappa + r]
        end

        mul!(Mu, Mmat, uvec)
        mul!(Ku, Kmat, uvec)
        mul!(Mv, Mmat, vvec)

        rhs_u .= a .* Mu .- b .* Ku .+ 2.0 .* Mv .+ 2.0 .* noise
        copyto!(uvec_new, rhs_u)
        ldiv!(facLHS, uvec_new)

        usum .= uvec_new .+ uvec
        mul!(Ku, Kmat, usum)

        rhs_v .= Mv .- b .* Ku .+ noise
        copyto!(vvec_new, rhs_v)
        ldiv!(facM, vvec_new)

        uvec, uvec_new = uvec_new, uvec
        vvec, vvec_new = vvec_new, vvec
    end

    return uvec
end
function EMpath_space(uh0, vh0, num_steps, Δt, facLHS, facM, Mmat, Kmat, noise_loads)
    a = 2 / Δt
    b = Δt / 2

    uvec = copy(get_free_dof_values(uh0))
    vvec = copy(get_free_dof_values(vh0))

    uvec_new = similar(uvec)
    vvec_new = similar(uvec)

    Mu = similar(uvec)
    Ku = similar(uvec)
    Mv = similar(uvec)
    usum = similar(uvec)

    rhs_u = similar(uvec)
    rhs_v = similar(uvec)

    for n in 1:num_steps
        noise = noise_loads[n]

        mul!(Mu, Mmat, uvec)
        mul!(Ku, Kmat, uvec)
        mul!(Mv, Mmat, vvec)

        rhs_u .= a .* Mu .- b .* Ku .+ 2.0 .* Mv .+ 2.0 .* noise
        copyto!(uvec_new, rhs_u)
        ldiv!(facLHS, uvec_new)

        usum .= uvec_new .+ uvec
        mul!(Ku, Kmat, usum)

        rhs_v .= Mv .- b .* Ku .+ noise
        copyto!(vvec_new, rhs_v)
        ldiv!(facM, vvec_new)

        uvec, uvec_new = uvec_new, uvec
        vvec, vvec_new = vvec_new, vvec
    end

    return uvec
end

EMpath_space (generic function with 1 method)

In [10]:
function runEMpath_stoch_time(uh0, vh0, T, Nref, kappas, Nsamp, Mmat, Kmat,
                              m0, m_ce, λ_ce, ce_h, Unoise, Bnoise)

    nk = length(kappas)
    Dtref = T / Nref
    ndofs = size(Mmat,1)

    partial_sums = [zeros(nk) for _ in 1:nthreads()]

    facM = lu(Mmat)

    Dt1 = Dtref
    LHSpos1 = (2 / Dt1) * Mmat + (Dt1 / 2) * Kmat
    facLHS1 = lu(LHSpos1)

    facLHS_list = Vector{typeof(facLHS1)}(undef, nk)
    for (i, kappa) in enumerate(kappas)
        Dt = kappa * Dtref
        LHSpos = (2 / Dt) * Mmat + (Dt / 2) * Kmat
        facLHS_list[i] = lu(LHSpos)
    end

    @threads for samp in 1:Nsamp
        tid = threadid()
        local_sum = partial_sums[tid]

        seed = 1337 + samp
        rng = Xoshiro(seed)

        noise_ref = build_reference_noise_loads(
            T, Nref, m0, m_ce, λ_ce, ce_h, Unoise, Bnoise, rng
        )

        uref = EMpath_stoch_time(
            uh0, vh0, T, Nref, 1, facLHS1, facM, Mmat, Kmat, noise_ref
        )

        Merr = zeros(ndofs)

        for (i, kappa) in enumerate(kappas)
            u = EMpath_stoch_time(
                uh0, vh0, T, Nref, kappa, facLHS_list[i], facM, Mmat, Kmat, noise_ref
            )

            erru = u - uref
            mul!(Merr, Mmat, erru)
            local_sum[i] += dot(erru, Merr)
        end
    end

    S_u = zeros(nk)
    for ps in partial_sums
        S_u .+= ps
    end

    return sqrt.(S_u ./ Nsamp)
end


function runEMpath_stoch_space(T, Nref, kappas, Nsamp)
    nk = length(kappas)

    N_list = [Nref; Int.(Nref ./ kappas)]

    n_time = 5
    Δt = 2.0^(-n_time)
    num_steps = Int(round(T / Δt))

    M_list      = Vector{Any}(undef, nk+1)
    K_list      = Vector{Any}(undef, nk+1)
    U_list      = Vector{Any}(undef, nk+1)
    V_list      = Vector{Any}(undef, nk+1)
    Unoise_list = Vector{Any}(undef, nk+1)
    Bnoise_list = Vector{Any}(undef, nk+1)
    uh0_list    = Vector{Any}(undef, nk+1)
    vh0_list    = Vector{Any}(undef, nk+1)

    order = 1
    reffe = ReferenceFE(lagrangian, Float64, order)

    for i in 1:(nk+1)
        N = N_list[i]
        h = 1 / N

        model = create_square_model(h)

        V = TestFESpace(model, reffe, conformity=:H1, dirichlet_tags="boundary")
        U_list[i] = TrialFESpace(V, 0.0)
        V_list[i] = V

        Vnoise = TestFESpace(model, reffe, conformity=:H1)
        Unoise_list[i] = TrialFESpace(Vnoise)

        Ω = Triangulation(model)
        dΩ = Measure(Ω, 2 * order)

        m(u,v) = ∫(u*v)dΩ
        k(u,v) = ∫(∇(u) ⋅ ∇(v))dΩ

        M_list[i] = assemble_matrix(m, U_list[i], V)
        K_list[i] = assemble_matrix(k, U_list[i], V)
        Bnoise_list[i] = assemble_matrix(m, Unoise_list[i], V)

        uh0_list[i] = interpolate_everywhere(x -> 0.0, U_list[i])
        vh0_list[i] = interpolate_everywhere(x -> 0.0, U_list[i])
    end

    facM_list = [lu(M_list[i]) for i in 1:(nk+1)]
    facLHS_list = [lu((2/Δt)*M_list[i] + (Δt/2)*K_list[i]) for i in 1:(nk+1)]

    partial_sums = [zeros(nk) for _ in 1:nthreads()]

    Mref = M_list[1]
    Uref = U_list[1]

    # CE built on finest mesh spacing
    ce_h = 1 / Nref
    σ = 1.0
    nu = 0.4
    p = 0.15

    m0 = Int(round(1/ce_h))
    m_ce, λ_ce = algorithm1(m0, σ, nu, p)

    @threads for samp in 1:Nsamp
        tid = threadid()
        local_sum = partial_sums[tid]

        rng = Xoshiro(1337 + samp)

        noise_loads_list = [Vector{Vector{Float64}}(undef, num_steps) for _ in 1:(nk+1)]

        for n in 1:num_steps
            Z = algorithm2(m0, m_ce, λ_ce, rng)

            for j in 1:(nk+1)
                noise_loads_list[j][n] = noise_load_from_grid(
                    Z, Δt, ce_h, Unoise_list[j], Bnoise_list[j]
                )
            end
        end

        uref = EMpath_space(
            uh0_list[1], vh0_list[1], num_steps, Δt,
            facLHS_list[1], facM_list[1], M_list[1], K_list[1],
            noise_loads_list[1]
        )

        Merr = similar(uref)

        for i in 1:nk
            idx = i + 1

            u = EMpath_space(
                uh0_list[idx], vh0_list[idx], num_steps, Δt,
                facLHS_list[idx], facM_list[idx], M_list[idx], K_list[idx],
                noise_loads_list[idx]
            )

            uh = FEFunction(U_list[idx], u)
            uh_ref = interpolate_everywhere(x -> uh(x), Uref)
            u_interp = get_free_dof_values(uh_ref)

            err = u_interp - uref
            mul!(Merr, Mref, err)
            local_sum[i] += dot(err, Merr)
        end
    end

    S = zeros(nk)
    for ps in partial_sums
        S .+= ps
    end

    return sqrt.(S ./ Nsamp)
end

runEMpath_stoch_space (generic function with 1 method)

In [ ]:
T = 1
kappas = [2,4,8,16,32,64,128,256,512,1024,2048]#[2,4,8,16,32,64,128,256,512,1024,2048,4096]
Nref = 2^13#2^15
Nsamp = 20
Dtref = T / Nref
dts = kappas .* Dtref

@time begin
    rms_u_time = runEMpath_stoch_time(
        uh0, vh0, T, Nref, kappas, Nsamp, Mmat, Kmat,
        m0, m_ce, λ_ce, ce_h, Unoise, Bnoise
    )
end



Nref_space = 2^5
kappas_space = [2,4,8,16]
Nsamp_space = 20
lenref = 1 / Nref_space
hs = kappas_space .* lenref

@time begin
    rms_u_space = runEMpath_stoch_space(T, Nref_space, kappas_space, Nsamp_space)
end




println("number of threads:")
println(nthreads())

println("----------------------------TIME----------------------------")
println("dt values:")
println(dts)
println("RMS errors:")
println(rms_u_time)
println("------------------------------------------------------------")

println("")

println("----------------------------SPACE----------------------------")
println("h values:")
println(hs)
println("RMS errors:")
println(rms_u_space)
println("-------------------------------------------------------------")

┌ Info: waiting for UmfpackLU's lock, it's safe to ignore this message.
└ see the documentation for Umfpack


In [ ]:
x = range(minimum(dts),maximum(dts), 100)
# log-log plot time
plot(dts, rms_u_time, xaxis=:log, yaxis=:log, marker=:cross, markercolor=:black, linecolor=:blue, label="RMS error")
plot!(x, 0.2*x.^1, xaxis=:log, yaxis=:log,linecolor=:red, label="Reference line")


In [ ]:
y = range(minimum(hs),maximum(hs),100)
# log-log plot space
plot(hs, rms_u_space, xaxis=:log, yaxis=:log, marker=:cross, markercolor=:black, linecolor=:blue, label="RMS error")
plot!(y, 0.2*y.^1, xaxis=:log, yaxis=:log,linecolor=:red, label="Reference line")